# Detecção de anomalias em telemetria de satélite — experimento inicial

Este notebook representa o **ponto de partida acadêmico** do minicurso
**“Do experimento à produção”**.

O objetivo científico aqui é propositalmente simples: treinar um classificador
para identificar amostras anômalas de uma telemetria **inteiramente sintética**.

> **Importante:** os valores e as regras usadas para gerar os dados são didáticos
> e não representam limites operacionais de nenhum satélite real.

Neste estágio, o foco está em responder apenas:

> **“Consigo treinar um modelo que funcione?”**

Nas próximas etapas do minicurso surgirão novas perguntas: reprodutibilidade,
rastreabilidade, serving, containerização, orquestração, CI/CD e cloud.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "telemetry.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "model.pkl"

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df.head()

## 1. Conhecendo os dados

Cada linha representa uma observação de telemetria. A coluna `anomaly` é o alvo
de classificação: `0` para operação nominal e `1` para amostra anômala.


In [ ]:
print(f"Amostras: {len(df):,}")
print(f"Período: {df['timestamp'].min()} até {df['timestamp'].max()}")
print()
print(df["anomaly"].value_counts())
print()
print(df["anomaly"].value_counts(normalize=True).rename("proportion"))

In [ ]:
ax = df.set_index("timestamp")[["battery_voltage", "battery_temperature"]].iloc[:500].plot(
    figsize=(12, 4)
)
ax.set_title("Exemplo das primeiras 500 observações")
ax.set_xlabel("tempo")
plt.show()

## 2. Preparando treino e teste

Neste primeiro experimento usaremos um `train_test_split` estratificado.
A simplicidade é intencional: o objetivo do minicurso não é discutir a melhor
estratégia de validação para telemetria temporal, e sim acompanhar a evolução
de um experimento até uma solução de engenharia.


In [ ]:
FEATURES = [
    "battery_voltage",
    "battery_current",
    "battery_temperature",
    "solar_panel_current",
    "bus_voltage",
    "attitude_error",
    "eclipse",
]

X = df[FEATURES]
y = df["anomaly"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Treino:", X_train.shape)
print("Teste :", X_test.shape)

## 3. Treinando o modelo

Usaremos um Random Forest por ser rápido, robusto e fácil de executar em
qualquer notebook ou notebook acadêmico comum, sem GPU.


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

## 4. Avaliação

In [ ]:
print(classification_report(y_test, pred, digits=3))

print("F1       :", round(f1_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall   :", round(recall_score(y_test, pred), 4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Matriz de confusão")
plt.show()

## 5. Salvando o modelo

Neste estágio, simplesmente salvamos o objeto treinado em um arquivo.

Isso funciona — mas já abre perguntas que motivarão as próximas etapas:

- Qual código gerou este arquivo?
- Quais hiperparâmetros foram usados?
- Qual métrica levou à escolha deste modelo?
- Como reproduzir exatamente o ambiente?
- Como outro sistema poderia consumir esta predição?
- Como automatizar treinamento e entrega?


In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)

print(f"Modelo salvo em: {MODEL_PATH}")